# purple-sectors · 02 · v1.1 (within-segment track evolution)
Recovery check for the new γ term, then the real 2026 fit. Runtime ≈ 15 min.

In [ ]:
# 1 · Drag the newest purple-sectors zip into the Files panel first, then run
import glob, io, os, zipfile
from google.colab import files
os.chdir('/content')
z = sorted(glob.glob('/content/purple-sectors*.zip'), key=os.path.getmtime)
assert z, 'No purple-sectors zip in /content. Drag it into the Files panel and wait for the upload.'
zipfile.ZipFile(z[-1]).extractall('/content')
os.chdir('/content/purple-sectors')
print('using', z[-1])

In [ ]:
# 2 · Install CmdStan (~5 min, once per Colab session)
%pip install -q cmdstanpy
import cmdstanpy
cmdstanpy.install_cmdstan(progress=True)

In [ ]:
# 3 · Load the real design
import os, sys
os.chdir('/content/purple-sectors')        # survives a Colab session restart
sys.path.insert(0, '/content/purple-sectors/src')
import numpy as np, pandas as pd
from cmdstanpy import CmdStanModel
from purple_sectors.prepare import load_push_laps, build_design
from purple_sectors.simulate import draw_params, simulate_laps

laps = load_push_laps('data/raw/quali_laps_2026.csv')
des = build_design(laps)
print(f'{des.N} push laps · {des.D} drivers · {des.C} cars · {des.E} events')

In [ ]:
# 4 · Helpers
model = CmdStanModel(stan_file='stan/quali_v11.stan')
HYPER = ['m', 'sigma_kappa', 'mu_gamma', 'sigma_gamma', 'sigma_car', 'sigma_dev', 'sigma_nu',
         'sigma_driver', 'beta_m', 'beta_a', 'sigma', 'loss']

def fit_model(y, seed=1):
    return model.sample(data=des.stan_data(y), chains=4, parallel_chains=4,
                        iter_warmup=1500, iter_sampling=3000, adapt_delta=0.95,
                        inits=0.5, seed=seed, show_progress=False)

def max_rhat(fit, vars):
    df = fit.draws_pd(vars=vars)
    cols = [c for c in df.columns if not c.endswith('__')]
    n_it = fit.num_draws_sampling
    x = df[cols].to_numpy().reshape(fit.chains, n_it, len(cols)).transpose(1, 0, 2)
    h = n_it // 2
    x = np.concatenate([x[:h], x[h:2 * h]], axis=1)
    W = x.var(axis=0, ddof=1).mean(axis=0)
    B = h * x.mean(axis=0).var(axis=0, ddof=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        r = np.sqrt(((h - 1) / h * W + B / h) / W)
    return np.nanmax(r)

def health(fit, name):
    div = int(fit.method_variables()['divergent__'].sum())
    rh = max_rhat(fit, HYPER + ['gamma', 'delta', 'theta', 'nu', 'tau'])
    print(f'{name:<16} divergences: {div:<4} max R-hat: {rh:.3f}   {"OK" if div == 0 and rh < 1.01 else "CHECK"}')

def cover(draws, true):
    lo, hi = np.percentile(draws, [5, 95], axis=0)
    return ((true >= lo) & (true <= hi)).mean()

In [ ]:
# 5 · Recovery: simulate with known parameters (incl. gamma), fit, check coverage
rng = np.random.default_rng(42)
TRUTH = dict(m=np.array([-0.55, -0.40]), sigma_kappa=0.15, mu_gamma=-0.4, sigma_gamma=0.15,
             sigma_car=1.5, sigma_dev=0.25, sigma_nu=0.30, sigma_driver=0.30,
             beta_m=0.8, beta_a=0.04, sigma=0.25, loss=0.35)
truth = draw_params(des, rng, v11=True, **TRUTH)
sim = fit_model(simulate_laps(des, truth, rng))
health(sim, 'simulated v1.1')
print(pd.Series({
    'gamma (within-segment)': cover(sim.stan_variable('gamma'), truth['gamma']),
    'driver skill δ': cover(sim.stan_variable('delta'), truth['delta']),
    'car trend θ': cover(sim.stan_variable('theta'), truth['theta']),
    'car × track ν': cover(sim.stan_variable('nu'), truth['nu']),
    'track evolution τ': cover(sim.stan_variable('tau')[:, :, 1:], truth['tau'][:, 1:]),
}, name='90% coverage').map('{:.0%}'.format).to_string())
l, m_, h = np.percentile(sim.stan_variable('mu_gamma'), [5, 50, 95])
print(f'mu_gamma truth -0.400 · posterior {m_:.3f} [{l:.3f}, {h:.3f}]')

## Real 2026 fit · run only if recovery shows 0 divergences and coverage ≈ 80–95%

In [ ]:
# 6 · Fit v1.1 on the real laps
y_real = des.laps['y'].to_numpy()
real = fit_model(y_real, seed=2026)
health(real, 'real 2026 v1.1')
for v in ['mu_gamma', 'sigma_gamma', 'sigma_driver', 'sigma', 'loss']:
    l, m_, h = np.percentile(real.stan_variable(v), [5, 50, 95])
    print(f'{v:<13} {m_:.3f}  [{l:.3f}, {h:.3f}]')
l, m_, h = np.percentile(real.stan_variable('car_share').mean(axis=1), [5, 50, 95])
print(f'car share     {m_:.3f}  [{l:.3f}, {h:.3f}]')

In [ ]:
# 7 · Export (upload posterior_v11.zip back to the chat)
KEEP = ['theta', 'nu', 'delta', 'tau', 'gamma', 'alpha', 'car_share', 'm', 'sigma_kappa',
        'mu_gamma', 'sigma_gamma', 'sigma_car', 'sigma_dev', 'sigma_nu', 'sigma_driver',
        'beta_m', 'beta_a', 'sigma', 'loss']
out = {v: real.stan_variable(v)[::6].astype('float32') for v in KEEP}
out['y_rep'] = real.stan_variable('y_rep')[::24].astype('float32')
out['log_lik'] = real.stan_variable('log_lik')[::12].astype('float32')
mv = real.method_variables()
out['divergent'] = mv['divergent__']; out['treedepth'] = mv['treedepth__']
np.savez_compressed('posterior_v11.npz', **out, drivers=np.array(des.drivers), cars=np.array(des.cars),
                    events=np.array(des.events), y=y_real, driver=des.driver, car=des.car,
                    event=des.event, seg=des.seg, t_seg=des.t_seg)
with zipfile.ZipFile('posterior_v11.zip', 'w') as z:
    z.write('posterior_v11.npz')
files.download('posterior_v11.zip')